# RVC Colab 训练入口

这个 notebook 使用 `rvc_training_data/colab/train_rvc_colab.py` 执行完整 RVC 训练流程：预处理、F0、HuBERT 特征、训练和 index 构建。

请先在本地用 `rvc-generate-dataset` 生成 30-60 分钟、单条 5-15 秒的数据集，并上传到 Google Drive。

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## 参数

`DATASET_ZIP` 和 `DATASET_DIR` 二选一：如果上传的是 zip，填写 `DATASET_ZIP`；如果已经是解压后的目录，填写 `DATASET_DIR`。

In [3]:
from pathlib import Path

REPO_DIR = Path('/content/RVC')
EXPERIMENT = 'leijun_zh_v2_48k'

# 填其中一个。zip 示例: /content/drive/MyDrive/zh_leijun_45m.zip
DATASET_ZIP = Path('/content/drive/MyDrive/zh_leijun_45m.zip')
DATASET_DIR = Path('/content/drive/MyDrive/rvc_datasets/zh_leijun_45m')

SAMPLE_RATE = '48k'
VERSION = 'v2'
IF_F0 = 1
F0_METHOD = 'rmvpe'
GPUS = '0'
BATCH_SIZE = 8
TOTAL_EPOCH = 200
SAVE_EVERY_EPOCH = 20

EXPORT_DIR = Path('/content/drive/MyDrive/rvc_models') / EXPERIMENT

## 安装 RVC

如果 `/content/RVC` 已存在，会复用现有目录。

In [ ]:
%cd /content
!test -d "$REPO_DIR" || git clone https://github.com/RVC-Project/Retrieval-based-Voice-Conversion-WebUI "$REPO_DIR"
%cd /content/RVC
!pip install -r requirements.txt
!python tools/download_models.py

/content
/content/RVC
Ignoring onnxruntime: markers 'sys_platform == "darwin"' don't match your environment
  Using cached aria2-0.0.1b0-py3-none-manylinux_2_17_x86_64.whl.metadata (28 kB)
  Using cached numba-0.56.4.tar.gz (2.4 MB)
  error: subprocess-exited-with-error
  
  × python setup.py egg_info did not run successfully.
  │ exit code: 1
  ╰─> See above for output.
  
  note: This error originates from a subprocess, and is likely not a problem with pip.
  Preparing metadata (setup.py) ... error
error: metadata-generation-failed

× Encountered error while generating package metadata.
╰─> See above for output.

note: This is an issue with the package mentioned above, not pip.
hint: See above for details.


## 准备训练脚本

把本仓库的 `rvc_training_data/colab/train_rvc_colab.py` 上传到 Colab 左侧文件区，或放到 Drive 的 `MyDrive/rvc_training_data/colab/`。下面的单元会自动查找这些位置。

In [ ]:
import shutil

SCRIPT_TARGET = REPO_DIR / 'rvc_training_data' / 'colab' / 'train_rvc_colab.py'
SCRIPT_TARGET.parent.mkdir(parents=True, exist_ok=True)
SCRIPT_CANDIDATES = [
    Path('/content/train_rvc_colab.py'),
    Path('/content/rvc_training_data/colab/train_rvc_colab.py'),
    Path('/content/drive/MyDrive/rvc_training_data/colab/train_rvc_colab.py'),
]

source = next((p for p in SCRIPT_CANDIDATES if p.exists()), None)
if source is not None:
    shutil.copy2(source, SCRIPT_TARGET)
elif not SCRIPT_TARGET.exists():
    raise FileNotFoundError('请上传 train_rvc_colab.py 到 /content/train_rvc_colab.py，或放到 Drive 的 MyDrive/rvc_training_data/colab/。')

print(f'Training script: {SCRIPT_TARGET}')

## 准备数据集

In [ ]:
import zipfile

if DATASET_ZIP.exists():
    DATASET_DIR.parent.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(DATASET_ZIP) as archive:
        archive.extractall(DATASET_DIR.parent)

if not DATASET_DIR.exists():
    raise FileNotFoundError(f'找不到数据集目录: {DATASET_DIR}')

audio_dir = DATASET_DIR / 'audio'
if not audio_dir.exists():
    audio_dir = DATASET_DIR

audio_files = [p for p in audio_dir.iterdir() if p.suffix.lower() in {'.mp3', '.wav', '.flac', '.m4a', '.ogg', '.aac', '.opus'}]
print(f'Dataset: {DATASET_DIR}')
print(f'Audio files: {len(audio_files)}')
if not audio_files:
    raise RuntimeError('数据集目录里没有音频文件。')

## 开始训练

In [ ]:
!python "$SCRIPT_TARGET" \
  --repo-dir "$REPO_DIR" \
  --dataset-dir "$DATASET_DIR" \
  --experiment "$EXPERIMENT" \
  --sample-rate "$SAMPLE_RATE" \
  --version "$VERSION" \
  --if-f0 "$IF_F0" \
  --f0-method "$F0_METHOD" \
  --gpus "$GPUS" \
  --batch-size "$BATCH_SIZE" \
  --total-epoch "$TOTAL_EPOCH" \
  --save-every-epoch "$SAVE_EVERY_EPOCH" \
  --save-latest 1 \
  --cache-gpu 0 \
  --save-every-weights 1

## 导出到 Drive

In [ ]:
import glob

EXPORT_DIR.mkdir(parents=True, exist_ok=True)
for pattern in [
    str(REPO_DIR / 'assets' / 'weights' / f'{EXPERIMENT}.pth'),
    str(REPO_DIR / 'logs' / EXPERIMENT / '*.index'),
    str(REPO_DIR / 'logs' / EXPERIMENT / 'train.log'),
    str(REPO_DIR / 'logs' / EXPERIMENT / 'colab_train_summary.json'),
]:
    for src in glob.glob(pattern):
        shutil.copy2(src, EXPORT_DIR / Path(src).name)

print(f'Exported files to: {EXPORT_DIR}')
print('\n'.join(str(p) for p in sorted(EXPORT_DIR.iterdir())))